## Dallas Bronze to Silver
Reads the latest file_location of the table from the metadata table, loads the Parquet snapshot, and overwrites the corresponding Silver Delta table.

In [0]:
%pip install databricks-labs-dqx
# method to restart
%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.1/775.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.1/606.1 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.2
    Not uninstalling pyyaml at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-aa35d773-eed6-417c-b6bd-247a1878464a
    Can't uninstall 'PyYAML'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-aa35d773-eed6-417c-b6bd-247a1878464a
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: dat

In [0]:
catalog_name  = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

print(f"catalog_name  : {catalog_name}")
print(f"bronze_schema : {bronze_schema}")
print(f"silver_schema : {silver_schema}")

catalog_name  : workspace
bronze_schema : final_project_bronze
silver_schema : final_project_silver


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType, LongType, IntegerType
from datetime import datetime, timezone
import re
from pyspark.sql.window import Window

run_ts = datetime.now(timezone.utc)

### Get Latest File Location per Table from Child Metadata

In [0]:
child_df = spark.table(f"{catalog_name}.{bronze_schema}.pipeline_metadata_child")

# Pick the most recent SUCCESS run per table
latest_df = (
    child_df
    .filter("status = 'SUCCESS'")
    .groupBy("table_name")
    .agg(F.max_by("file_location", "execution_time").alias("file_location"))
)

latest_rows = latest_df.collect()
print(f"Tables to promote to Silver: {[r.table_name for r in latest_rows]}")

Tables to promote to Silver: ['Food_Inspections_Chicago', 'Food_Inspections_Dallas']


### Load Bronze Parquet → DQX → Transformations → Write Silver Delta (overwrite)

In [0]:
# load dallas file
table_name = latest_rows[1].table_name
file_loc   = latest_rows[1].file_location

print(f"  Processing: {table_name}")
print(f"  Source: {file_loc}")

dallas_df = spark.read.parquet(file_loc)

  Processing: Food_Inspections_Dallas
  Source: /Volumes/workspace/final_project_bronze/food_inspection/food_inspections_dallas/2026/04/12/food_inspections_dallas_20260412_172350.parquet


In [0]:
columns_to_drop = ["Street Number", "Street Name", "Street Direction", "Street Type", "Street Unit", "Inspection Month", "Inspection Year"]

dallas_df = dallas_df.drop(*columns_to_drop)

In [0]:
# change column names, remove spaces
dallas_df = dallas_df.toDF(*[
    re.sub(r"\s+", "_", col.lower().replace("-", ""))
    for col in dallas_df.columns
])

dallas_df = dallas_df\
    .withColumnRenamed("restaurant_name", "establishment_name")

dallas_df = dallas_df.withColumn("zip_code", F.col("zip_code").try_cast("string"))

points_cols = [c for c in dallas_df.columns if c.startswith("violation_points_")]
dallas_df = dallas_df.drop(*points_cols)

In [0]:
# create inspection_id column
start_value = 11111111
dallas_df = dallas_df.withColumn("inspection_id", F.monotonically_increasing_id()+start_value)

In [0]:
# create latitude, longitude columns from location column
dallas_df = dallas_df.withColumn(
    "lat_long_clean",
    F.when(F.col("lat_long_location").isNotNull(),
           F.regexp_replace("lat_long_location", "[()]", ""))
)

dallas_df = dallas_df.withColumn(
    "latitude",
    F.when(
        F.col("lat_long_clean").isNotNull() & 
        (F.size(F.split("lat_long_clean", ",")) == 2),
        F.split("lat_long_clean", ",").getItem(0).try_cast("double")
    )
).withColumn(
    "longitude",
    F.when(
        F.col("lat_long_clean").isNotNull() & 
        (F.size(F.split("lat_long_clean", ",")) == 2),
        F.trim(F.split("lat_long_clean", ",").getItem(1)).try_cast("double")
    )
).drop("lat_long_clean")

dallas_df = dallas_df.withColumnRenamed("lat_long_location", "location")

In [0]:
# clean up string columns
for col_name, dtype in dallas_df.dtypes:
    if dtype == "string":
        dallas_df = dallas_df\
                        .withColumn(col_name, F.trim(F.col(col_name)))\
                        .withColumn(col_name, F.upper(F.col(col_name)))

In [0]:
violation_cols = [f"violation_description_{i}" for i in range(1, 26)]

dallas_df = dallas_df.withColumn(
    "violation_count",
    sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in violation_cols)
)

In [0]:
dallas_df = dallas_df.withColumn(
    "inspection_result",
    F.when(F.col("inspection_score") > 80, "PASS")
     .when((F.col("inspection_score") > 70) & (F.col("inspection_score") <= 80), "PASS W/ CONDITIONS")
     .when((F.col("inspection_score") > 0) & (F.col("inspection_score") <= 70), "FAIL")
     .when(F.col("inspection_score") == 0, "NO ENTRY")
     .otherwise(None)
)

In [0]:
violation_cols = []
for i in range(1, 26):
    violation_cols += [
        f"violation_detail_{i}",
        f"violation_description_{i}",
        f"violation_memo_{i}"
    ]

dallas_df = dallas_df.withColumn(
    "has_urgent_or_critical",
    F.expr(f"""
        exists(array({",".join(violation_cols)}), x ->
            coalesce(x, '') LIKE '%URGENT%' OR
            coalesce(x, '') LIKE '%CRITICAL%'
        )
    """)
)

### Data Profiling

### Databricks DQX library modules
- **WorkspaceClient** Authenticate the Databricks SDK for Python with your Databricks account or workspace to perform data quality checking with DQX
- **DQEngine** Run the data quality checks

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

In [0]:
# The engine requires a Databricks workspace client for authentication and interaction with the Databricks workspace
ws = WorkspaceClient()

non_null_cols = ["establishment_name", "inspection_date", "inspection_type", "zip_code", "violation_description_1"]

non_null_checks = [
    {
        "criticality": "error",
        "name": f"{col}_is_null",
        "check": {
            "function": "sql_expression",
            "arguments": {
                "expression": f"{col} is not null",
                "msg": f"{col} must not be null",
            },
        },
    }
    for col in non_null_cols
]

inspection_checks = [
    {
        "criticality": "error",
        "name": "inspection_score_not_in_range",
        "check": {
            "function": "sql_expression",
            "arguments": {
                "expression": "inspection_score >= 0 and inspection_score <= 100",
                "msg":  "inspection_score must be between 0 and 100",
            },
        },
    },
    {
        "criticality": "error",
        "name": "high_score_low_violations",
        "check": {
            "function": "sql_expression",
            "arguments": {
                "expression": "NOT (inspection_score >= 90 AND violation_count > 3)",
                "msg": "If inspection_score >= 90, violations must be <= 3",
            },
        },
    },
    {
        "criticality": "error",
        "name": "pass_cannot_have_urgent_or_critical_violations",
        "check": {
            "function": "sql_expression",
            "arguments": {
                "expression": "NOT (inspection_result LIKE 'PASS%' AND   has_urgent_or_critical = true)",
                "msg": "PASS inspections cannot contain Urgent or Critical violations in any field",
            },
        },
    }
]

all_checks = non_null_checks + inspection_checks

# apply the checks generated based on profile to validate
dqengine = DQEngine(ws)
valid_dallas_df, invalid_dallas_df = dqengine.apply_checks_by_metadata_and_split(dallas_df, non_null_checks)

In [0]:
passed_count = valid_dallas_df.count()
failed_count = invalid_dallas_df.count()
print(f"  Passed: {passed_count} | Quarantined: {failed_count}")

# Write quarantine records
if failed_count > 0:
    data_cols = [c for c in invalid_dallas_df.columns if not c.startswith("_")]
    quarantine_out = (
        invalid_dallas_df
        .withColumn("table_name",     F.lit(table_name))
        .withColumn("run_timestamp",  F.lit(run_ts).cast(TimestampType()))
        .withColumn("_error_details", F.to_json(F.col("_errors")))
        .withColumn("_row_data",      F.to_json(F.struct(*[F.col(c) for c in data_cols])))
        .select("table_name", "run_timestamp", "_error_details", "_row_data")
    )
    quarantine_out.write.format("delta").mode("append").saveAsTable(
        f"{catalog_name}.{silver_schema}.dallas_quarantine"
    )
    print(f"{failed_count} records written to dallas quarantine.")

dqx_log_rows = []

dqx_log_rows.append((
    table_name,
    run_ts,
    passed_count,
    failed_count,
    run_ts.date()
))

  Passed: 72394 | Quarantined: 6590
6590 records written to dallas quarantine.


### Write DQX Execution Log

In [0]:
log_schema = StructType([
    StructField("table_name",     StringType(),    True),
    StructField("run_timestamp",  TimestampType(), True),
    StructField("passed_records", LongType(),      True),
    StructField("failed_records", LongType(),      True),
    StructField("created_date",   DateType(),      True),
])

log_df = spark.createDataFrame(dqx_log_rows, log_schema)
log_df.write.format("delta").mode("append").saveAsTable(
    f"{catalog_name}.{silver_schema}.dqx_execution_log"
)

print("\nDQX execution log written.")
display(log_df)


DQX execution log written.


table_name,run_timestamp,passed_records,failed_records,created_date
Food_Inspections_Dallas,2026-04-11T15:58:36.541Z,72394,6590,2026-04-11


In [0]:
# regex cleaning
# remove special characters from name columns
cols = ["establishment_name", "street_address"]

for c in cols:
    valid_dallas_df = valid_dallas_df\
                        .withColumn(c, F.regexp_replace(F.col(c), r"[\\.,']", ""))
                    

In [0]:
# create city column
valid_dallas_df = valid_dallas_df\
                    .withColumn("city", F.lit("DALLAS"))\
                    .withColumn("license_number", F.lit("9999999"))\
                    .withColumn("aka_establishment_name", F.col("establishment_name"))

# create establishment id
valid_dallas_df = valid_dallas_df.withColumn(
    "establishment_id", F.sha2( F.concat_ws("||",
            F.coalesce(F.col("establishment_name"), F.lit("")),
            F.coalesce(F.col("street_address"), F.lit("")),
            F.coalesce(F.col("city"), F.lit("")),
            F.coalesce(F.col("zip_code"), F.lit(""))
        ), 256
    )
)

In [0]:
# create the additional columns
valid_dallas_df = valid_dallas_df.withColumn("risk",
    F.when(F.col("inspection_score") > 80, "RISK (LOW)")
     .when((F.col("inspection_score") > 70) & (F.col("inspection_score") <= 80), "RISK (MEDIUM)")
     .when((F.col("inspection_score") > 0) & (F.col("inspection_score") <= 70), "RISK (HIGH)")
     .when(F.col("inspection_score") == 0, "ALL")
     .otherwise(None)
)

valid_dallas_df = valid_dallas_df.withColumn(
    "facility_type",
    F.when(F.col("establishment_name").contains("STORE"), "STORE")
     .when(F.col("establishment_name").contains("SCHOOL"), "SCHOOL")
     .when(F.col("establishment_name").contains("ELEMENTARY"), "ELEMENTARY SCHOOL")
     .when(F.col("establishment_name").contains("ACADEMY"), "ACADEMY")
     .when(F.col("establishment_name").contains("GROCERY"), "GROCERY")
     .when(F.col("establishment_name").contains("MART"), "MART")
     .when(F.col("establishment_name").contains("MARKET"), "MARKET")
     .when(F.col("establishment_name").contains("KITCHEN"), "KITCHEN")
     .when(F.col("establishment_name").contains("DINNING"), "DINNING")
     .when(F.col("establishment_name").contains("EATRY"), "EATRY")
     .when(F.col("establishment_name").contains("BAKERY"), "BAKERY")
     .when(F.col("establishment_name").contains("GYM"), "GYM")
     .when(F.col("establishment_name").contains("RECREATION"), "RECREATION CENTER")
     .when(F.col("establishment_name").contains("CLUB"), "CLUB")
     .when(F.col("establishment_name").contains("BAR"), "BAR")
     .when(F.col("establishment_name").contains("HOSTEL"), "HOSTEL")
     .when(F.col("establishment_name").contains("HOTEL"), "HOTEL")
     .when(F.col("establishment_name").contains("STADIUM"), "STADIUM")
     .when(F.col("establishment_name").contains("HOSPITAL"), "HOSPITAL")
     .otherwise("RESTAURANT")
)

In [0]:
# drop unwanted columns
valid_dallas_df = valid_dallas_df \
                    .drop(
                        "violation_count",
                        "has_urgent_or_critical")

In [0]:
# unpivot the dataset
stack_expr = "stack(25, " + ", ".join([
    f"{i}, violation_detail_{i}, violation_description_{i}, violation_memo_{i}"
    for i in range(1, 26)
]) + ") as (violation_number, detail, description, memo)"

df_long = valid_dallas_df.select(
    'inspection_id',
    'establishment_id',
    'establishment_name',
    'inspection_type',
    'inspection_date',
    'inspection_score',
    'street_address',
    'zip_code',
    'location',
    'latitude',
    'longitude',
    'inspection_result',
    'city',
    'license_number',
    'aka_establishment_name',
    'risk',
    'facility_type',
    F.expr(stack_expr)
).filter("detail IS NOT NULL OR description IS NOT NULL")

In [0]:
# split and recombine the violation description and details

# violation description column
df_long = df_long.withColumn(
    "detail_clean",
    F.regexp_replace("description", r"^[\*\s]+", "")
)

df_long = df_long.withColumn(
    "detail_code",
    F.regexp_extract("detail_clean", r"^(\d+)", 1)
).withColumn(
    "detail_text",
    F.regexp_replace("detail_clean", r"^\d+\s*", "")
)

# violation detail column
df_long = df_long.withColumn(
    "desc_clean",
    F.regexp_replace("detail", r"^[^A-Za-z0-9]+", "")
)

df_long = df_long.withColumn(
    "desc_code",
    F.regexp_extract("desc_clean", r"^([A-Za-z\. ]*\d[\d\.\-]*)", 1)
).withColumn(
    "desc_text",
    F.regexp_replace("desc_clean", r"^[A-Za-z\. ]*\d[\d\.\-]*\s*", "")
)

# recombine columns
df_long = df_long.withColumn(
    "violation_code",
    F.concat_ws(" - ", "detail_code", "desc_code")
).withColumn(
    "violation_description",
    F.concat_ws(" - ", "detail_text", "desc_text")
)

# drop intermediate columns
df_long = df_long.drop(
    "detail_clean", "detail_code", "detail_text", "desc_clean", "desc_code", "desc_text", "violation_number", "detail", "description"
)

final_df = df_long.withColumnRenamed("memo", "violation_comments")

In [0]:
final_df = final_df.dropDuplicates()

final_df = final_df\
                .withColumn("source_table", F.lit(table_name))\
                .withColumn("load_date", F.lit(run_ts))

In [0]:
final_df.columns

['inspection_id',
 'establishment_id',
 'establishment_name',
 'inspection_type',
 'inspection_date',
 'inspection_score',
 'street_address',
 'zip_code',
 'location',
 'latitude',
 'longitude',
 'inspection_result',
 'city',
 'license_number',
 'aka_establishment_name',
 'risk',
 'facility_type',
 'violation_comments',
 'violation_code',
 'violation_description',
 'source_table',
 'load_date']

In [0]:
# Write to Silver
(
    final_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{silver_schema}.dallas_food_inspection")
)
print(f"  Silver table dallas_food_inspection written.")

  Silver table dallas_food_inspection written.
